# 00_Data_Audit – Findings and Next Steps

## <b> Adrian Vazquez </b>
What López de Prado calls 'Data Validation & Research Dataset Construction'. 

The strategy has already been written as if the data were already clean, but I don't know that yet.

---
## Objective

Before validating any hypothesis or building a backtesting engine, a data audit was conducted to verify that the dataset is consistent with the strategy specifications and suitable for quantitative research.

The goal of this phase was not to evaluate profitability, but rather to understand the structure, quality, and limitations of the available data.

---

# Dataset Summary

### Nasdaq-100 Dataset

* ~47.9 million observations
* 95 unique tickers
* Period covered:

  * Start: 2021-06-01 04:00:00
  * End: 2026-02-04 19:59:00
* Intraday OHLCV data with additional metadata:

  * VWAP
  * Transactions
  * Market Cap
  * SIC Sector Information

### QQQ Dataset

* ~1.0 million observations
* Same date range as Nasdaq dataset
* Will be used later as the benchmark (Buy & Hold QQQ).

---


In [2]:
import pandas as pd
from pathlib import Path
import sys
import polars as pl
import pyarrow
import gc
sys.path.append(str(Path("..")))
from src.ingest.open_parquet import abrir_parquet_data_polar
from src.ingest.open_parquet import abrir_parquet_data_pandas




In [ ]:
# Cargar el primer archivo
df_nasdaq_100 = abrir_parquet_data_pandas("nasdaq100_with_meta.parquet")
df_QQQ = abrir_parquet_data_pandas('QQQ_1m.parquet')


Cargando exitosamente: nasdaq100_with_meta.parquet
Cargando exitosamente: QQQ_1m.parquet


In [6]:
print(df_nasdaq_100["date"].dt.tz)

None


In [7]:
print(df_QQQ["date"].dt.tz)

None


In [8]:
df_nasdaq_100["symbol"].nunique()

95

In [14]:
#Todos existen durante todo el período?
coverage = (df_nasdaq_100.groupby("symbol").agg(start=("date","min"),end=("date","max")))
coverage

,start,end
symbol,,
AAPL,2021-06-01 04:00:00,2026-02-04 19:59:00
ABNB,2021-06-01 04:02:00,2026-02-04 19:10:00
ADBE,2021-06-01 08:01:00,2026-02-04 19:59:00
ADI,2021-06-01 07:34:00,2026-02-04 18:59:00
ADP,2021-06-01 08:07:00,2026-02-04 19:30:00
...,...,...
WDAY,2021-06-01 09:30:00,2026-02-04 19:15:00
WDC,2021-06-01 08:00:00,2026-02-04 19:58:00
WMT,2021-06-01 07:27:00,2026-02-04 19:59:00


In [11]:
sample = (df_nasdaq_100.query("symbol=='AAPL'").sort_values("date"))
sample["delta"] = sample["date"].diff()
sample["delta"].value_counts().head(20)

delta
0 days 00:01:00    806638
0 days 00:02:00     52843
0 days 00:03:00     20319
0 days 00:04:00      9813
0 days 00:05:00      5405
0 days 00:06:00      3188
0 days 00:07:00      2055
0 days 00:08:00      1268
0 days 00:09:00       886
0 days 08:01:00       791
0 days 00:10:00       603
0 days 00:11:00       417
0 days 00:12:00       335
0 days 00:13:00       221
2 days 08:01:00       190
0 days 00:14:00       168
0 days 00:15:00       138
0 days 00:16:00        98
0 days 08:02:00        75
0 days 00:17:00        75
Name: count, dtype: int64

In [12]:
df_nasdaq_100.groupby("symbol")["market_cap"].nunique()

symbol
AAPL    1
ABNB    1
ADBE    1
ADI     1
ADP     1
       ..
WDAY    1
WDC     1
WMT     1
XEL     1
ZS      1
Name: market_cap, Length: 95, dtype: int64

In [13]:
(df_nasdaq_100[["symbol","sic_description"]].drop_duplicates().groupby("sic_description").size().sort_values(ascending=False))

sic_description
SERVICES-PREPACKAGED SOFTWARE                                   13
SEMICONDUCTORS & RELATED DEVICES                                12
PHARMACEUTICAL PREPARATIONS                                      4
SERVICES-BUSINESS SERVICES, NEC                                  4
SERVICES-COMPUTER PROCESSING & DATA PREPARATION                  3
SERVICES-COMPUTER PROGRAMMING, DATA PROCESSING, ETC.             3
BIOLOGICAL PRODUCTS, (NO DISGNOSTIC SUBSTANCES)                  2
COMPUTER PERIPHERAL EQUIPMENT, NEC                               2
SERVICES-COMPUTER PROGRAMMING SERVICES                           2
MOTOR VEHICLES & PASSENGER CAR BODIES                            2
RETAIL-VARIETY STORES                                            2
CABLE & OTHER PAY TELEVISION SERVICES                            2
COMPUTER STORAGE DEVICES                                         2
ELECTRIC & OTHER SERVICES COMBINED                               2
BEVERAGES                                     

# Key Findings

## 1. Timestamps are Timezone-Naive

Both datasets contain timezone-naive timestamps.

Since the strategy explicitly trades between:

* 09:30 AM ET
* 04:00 PM ET

it is necessary to define a timezone convention before strategy implementation.

At this stage, timestamps will be treated consistently across all assets, but timezone handling must be verified before backtesting.

---

## 2. Dataset Contains Pre-Market and After-Hours Data

Observed timestamps range from approximately:

* 04:00 AM
* 08:00 PM

which indicates the presence of:

* Pre-market trading
* Regular trading session
* After-hours trading

The strategy specification only allows trading during Regular Trading Hours (RTH).

Therefore, future signal generation and execution logic must be restricted to:

09:30 AM – 04:00 PM ET.

---

## 3. Nasdaq Universe Contains 95 Stocks

The dataset contains 95 unique symbols instead of the expected 100.

This does not prevent strategy development, but should be documented as a potential universe discrepancy.

For the purposes of this project, the available universe will be treated as the research universe.

---

## 4. Market Capitalization is Static

For every ticker:

market_cap.nunique() = 1

Therefore:

* Market capitalization is a static attribute.
* Leader/Follower classification can be computed once and reused.
* No historical market-cap reconstruction is required.

This simplifies the implementation considerably.

---

## 5. Sector Information is Available

The SIC classification appears sufficiently granular.

Examples:

* Software (~13 stocks)
* Semiconductors (~12 stocks)

This allows the construction of same-sector pairs as required by the strategy.

---

## 6. Intraday Data is Not Perfectly Regular

The majority of observations occur at 1-minute intervals, but gaps are present.

Examples observed:

* 1 minute
* 2 minutes
* 3 minutes
* 4 minutes
* larger gaps overnight and weekends

This means the dataset should not be assumed to be a perfectly regular time series.

Any lead-lag analysis performed directly on raw timestamps may introduce noise or synchronization issues.

---

# Conclusions

The dataset appears suitable for strategy research.

The two most important findings are:

1. The universe contains sufficient sector information to construct same-sector pairs.
2. Intraday observations are not perfectly synchronized and must be standardized before hypothesis testing.

No major data-quality issue was identified that would prevent continuing with the research process.

---

# Next Steps

The project objective remains unchanged:

Validate whether the proposed Leader-Follower strategy contains a genuine lead-lag effect before building a full backtest.

To achieve this, only the following preparation steps are required:

### Step 1

Standardize all price series to a common 1-minute grid.

This will ensure:

* Consistent timestamps across stocks.
* Reliable lead-lag measurements.
* Comparable return calculations.

### Step 2

Restrict analysis to Regular Trading Hours (09:30–16:00).

This aligns the research dataset with the strategy specification.

### Step 3

Compute 1-minute returns for each stock.

These returns will become the primary input for hypothesis testing.


## Research dataset construction
   - 1.1 Resample 1 mim
   - 1.2 Filter RTH (09:30-16:00)
   - 1.3 Returns 1m

In [3]:
# Standardize all price series to a common 1-minute grid
# what do we want to achieve? 
# We want all assets to have exactly the same time interval : avoid artifitial noise  
# Cargar archivos
df_nasdaq_100 = abrir_parquet_data_polar("nasdaq100_with_meta.parquet")
df_QQQ = abrir_parquet_data_polar('QQQ_1m.parquet')

# step 1. regular trading hour
df_nasdaq_100_rth = (df_nasdaq_100.filter((pl.col("date").dt.time() >= pl.time(9, 30)) & (pl.col("date").dt.time() <= pl.time(16, 0))))

# step 2.  avoid tickers by period 
# avoid tickets by period 
AVOID_TICKERS_BY_PERIOD = {
    "2021-01-01/2021-12-31": ["ABNB", "APP", "ARM", "AXON", "CCEP", "CEG", "CRWD", "DASH", "DDOG", "ENPH", "FER", "FTNT", "GEHC", "GFS", "INSM", "LCID", "LIN", "MDB", "META", "MPWR", "MSTR", "MTCH", "NTES", "ON", "PLTR", "RIVN", "SHOP", "STX", "TCOM", "TTD", "WBD", "WDC", "WMT", "ZS"],
    "2022-01-01/2022-12-31": ["ALNY", "APP", "ARM", "AXON", "CEG", "DASH", "FER", "GEHC", "GFS", "INSM", "LIN", "MDB", "META", "MPWR", "MSTR", "NTES", "ON", "PLTR", "RIVN", "SHOP", "STX", "TCOM", "TTD", "WBD", "WDC", "WMT", "ZS"],
    "2023-01-01/2023-12-31": ["ALNY", "APP", "ARM", "AXON", "CCEP", "DASH", "FER", "GEHC", "INSM", "LIN", "MDB", "MPWR", "MSTR", "ON", "PLTR", "SHOP", "STX", "TTD", "WDC", "WMT", "ZS"],
    "2024-01-01/2024-12-31": ["ALNY", "APP", "ARM", "ATVI", "AXON", "BIIB", "CERN", "CHKP", "DOCU", "EBAY", "ENPH", "FB", "FER", "FISV", "FOX", "FOXA", "INSM", "JD", "LCID", "LIN", "MPWR", "MSTR", "MXIM", "NTES", "OKTA", "PLTR", "PTON", "RIVN", "SGEN", "SHOP", "SPLK", "STX", "SWKS", "TCOM", "TTD", "VRSN", "WDC", "WMT", "XLNX", "ZM"],
    "2025-01-01/2025-12-21": ["ALGN", "ATVI", "BIIB", "CERN", "CHKP", "DOCU", "DLTR", "EBAY", "ENPH", "FB", "FER", "FISV", "FOX", "FOXA", "INSM", "JD", "LCID", "MNST", "MPWR", "MTCH", "MXIM", "NTES", "OKTA", "ON", "PTON", "RIVN", "SGEN", "SHOP", "SIRI", "SPLK", "STX", "SWKS", "TCOM", "TTD", "VRSN", "WBA", "WDC", "WMT", "XLNX", "ZM"],
    "2025-12-22/2026-01-19": ["ALGN", "ATVI", "BIIB", "CDW", "CERN", "CHKP", "DOCU", "DLTR", "EBAY", "ENPH", "FB", "FISV", "FOX", "FOXA", "GFS", "JD", "LCID", "LULU", "MRNA", "MTCH", "MXIM", "NTES", "OKTA", "ON", "PTON", "RIVN", "SGEN", "SHOP", "SIRI", "SPLK", "SWKS", "TCOM", "TTD", "VRSN", "WBA", "WMT", "XLNX", "ZM"],
    "2026-01-20/2026-12-31": ["ALGN", "ATVI", "AZN", "BIIB", "CDW", "CERN", "CHKP", "DOCU", "DLTR", "EBAY", "ENPH", "FB", "FISV", "FOX", "FOXA", "GFS", "JD", "LCID", "LULU", "MRNA", "MTCH", "MXIM", "NTES", "OKTA", "ON", "PTON", "RIVN", "SGEN", "SHOP", "SIRI", "SPLK", "SWKS", "TCOM", "TTD", "VRSN", "WBA", "XLNX", "ZM"],
}

def apply_avoid_tickers_polars(lf, avoid_dict):

    for period, tickers in avoid_dict.items():

        start_str, end_str = period.split("/")

        lf = lf.filter(
            ~((pl.col("date") >= pl.datetime( int(start_str[:4]), int(start_str[5:7]), int(start_str[8:10])))
                &
                (pl.col("date") <= pl.datetime(int(end_str[:4]), int(end_str[5:7]), int(end_str[8:10])))
                &
                (pl.col("symbol").is_in(tickers))
            ))

    return lf

df_nasdaq_100_research = apply_avoid_tickers_polars(df_nasdaq_100_rth,AVOID_TICKERS_BY_PERIOD)


Cargando: ..\data\nasdaq100_with_meta.parquet
Cargando: ..\data\QQQ_1m.parquet


In [4]:
out_dir = Path("../data/rth_1min_by_symbol")
out_dir.mkdir(exist_ok=True)

symbols = (df_nasdaq_100_research.select("symbol").unique().collect().get_column("symbol").to_list())
len(symbols)

95

In [5]:
def resample_symbol_day_rth_to_1min(df_symbol):
    df_symbol = df_symbol.sort_values("date").copy()
    df_symbol["trading_day"] = df_symbol["date"].dt.date

    out = []

    for _, df_day in df_symbol.groupby("trading_day", observed=True):
        symbol = df_day["symbol"].iloc[0]
        market_cap = df_day["market_cap"].iloc[0]
        sic_description = df_day["sic_description"].iloc[0]
        company_name = df_day["company_name"].iloc[0]

        day = pd.Timestamp(df_day["date"].iloc[0].date())
        full_index = pd.date_range(start=day + pd.Timedelta(hours=9, minutes=30),end=day + pd.Timedelta(hours=16),freq="1min")

        df_day = df_day.set_index("date")

        resampled = df_day.resample("1min").agg({
            "open": "last",
            "high": "last",
            "low": "last",
            "close": "last",
            "vwap": "last",
            "volume": "sum",
            "transactions": "sum",
        })

        resampled = resampled.reindex(full_index)

        price_cols = ["open", "high", "low", "close", "vwap"]
        flow_cols = ["volume", "transactions"]

        resampled[price_cols] = resampled[price_cols].ffill()
        resampled[flow_cols] = resampled[flow_cols].fillna(0)

        # por si el primer minuto del día venía vacío
        resampled[price_cols] = resampled[price_cols].bfill()

        resampled["symbol"] = symbol
        resampled["market_cap"] = market_cap
        resampled["sic_description"] = sic_description
        resampled["company_name"] = company_name

        resampled = resampled.reset_index().rename(columns={"index": "date"})
        out.append(resampled)

    return pd.concat(out, ignore_index=True)

In [6]:
# step 3. resample
for i, sym in enumerate(symbols, start=1):
    print(f"{i}/{len(symbols)} - {sym}")

    df_sym = (df_nasdaq_100_research.filter(pl.col("symbol") == sym).collect().to_pandas())

    df_sym_1min = resample_symbol_day_rth_to_1min(df_sym)

    df_sym_1min.to_parquet(out_dir / f"{sym}_rth_1min.parquet",index=False)
    del df_sym, df_sym_1min
    gc.collect()

1/95 - CSGP


KeyboardInterrupt: 

In [7]:
test = pd.read_parquet(out_dir / "AAPL_rth_1min.parquet")
test["trading_day"] = test["date"].dt.date
test.groupby("trading_day").size().describe()

count    1176.0
mean      391.0
std         0.0
min       391.0
25%       391.0
50%       391.0
75%       391.0
max       391.0
dtype: float64

In [8]:
records = []

for file in Path("../data/rth_1min_by_symbol").glob("*.parquet"):

    df = pd.read_parquet(file)

    records.append({
        "symbol": df["symbol"].iloc[0],
        "market_cap": df["market_cap"].iloc[0],
        "sector": df["sic_description"].iloc[0],
        "company_name": df["company_name"].iloc[0]
    })

universe_df = pd.DataFrame(records)

print(universe_df.shape)

universe_df.head()

(95, 4)


,symbol,market_cap,sector,company_name
0,AAPL,4.059188e+12,ELECTRONIC COMPUTERS,Apple Inc.
1,ABNB,7.555724e+10,SERVICES-TO DWELLINGS & OTHER BUILDINGS,"Airbnb, Inc. Class A Common Stock"
2,ADBE,1.148210e+11,SERVICES-PREPACKAGED SOFTWARE,Adobe Inc.
3,ADI,1.565761e+11,SEMICONDUCTORS & RELATED DEVICES,"Analog Devices, Inc."
4,ADP,9.448855e+10,SERVICES-COMPUTER PROCESSING & DATA PREPARATION,Automatic Data Processing


In [9]:
universe_df.groupby("sector").size().sort_values(ascending=False)

sector
SERVICES-PREPACKAGED SOFTWARE                                   13
SEMICONDUCTORS & RELATED DEVICES                                12
PHARMACEUTICAL PREPARATIONS                                      4
SERVICES-BUSINESS SERVICES, NEC                                  4
SERVICES-COMPUTER PROCESSING & DATA PREPARATION                  3
SERVICES-COMPUTER PROGRAMMING, DATA PROCESSING, ETC.             3
BIOLOGICAL PRODUCTS, (NO DISGNOSTIC SUBSTANCES)                  2
COMPUTER PERIPHERAL EQUIPMENT, NEC                               2
SERVICES-COMPUTER PROGRAMMING SERVICES                           2
MOTOR VEHICLES & PASSENGER CAR BODIES                            2
RETAIL-VARIETY STORES                                            2
CABLE & OTHER PAY TELEVISION SERVICES                            2
COMPUTER STORAGE DEVICES                                         2
ELECTRIC & OTHER SERVICES COMBINED                               2
BEVERAGES                                              

In [10]:
universe_df.sort_values("market_cap", ascending=False).head(20)

,symbol,market_cap,sector,company_name
63,NVDA,4.233688e+12,SEMICONDUCTORS & RELATED DEVICES,Nvidia Corp
0,AAPL,4.059188e+12,ELECTRONIC COMPUTERS,Apple Inc.
40,GOOG,4.022414e+12,"SERVICES-COMPUTER PROGRAMMING, DATA PROCESSING...",Alphabet Inc. Class C Capital Stock
39,GOOGL,4.018794e+12,"SERVICES-COMPUTER PROGRAMMING, DATA PROCESSING...",Alphabet Inc. Class A Common Stock
59,MSFT,3.075621e+12,SERVICES-PREPACKAGED SOFTWARE,Microsoft Corp
11,AMZN,2.490713e+12,RETAIL-CATALOG & MAIL-ORDER HOUSES,Amazon.Com Inc
85,TSLA,1.523525e+12,MOTOR VEHICLES & PASSENGER CAR BODIES,"Tesla, Inc. Common Stock"
14,AVGO,1.460549e+12,SEMICONDUCTORS & RELATED DEVICES,Broadcom Inc. Common Stock
92,WMT,1.020181e+12,RETAIL-VARIETY STORES,Walmart Inc. Common Stock
13,ASML,5.197802e+11,NaN,ASML Holding NV


In [11]:
universe_df["sector"].isna().sum()

np.int64(4)

In [12]:
universe_df.loc[universe_df["sector"].isna()]

,symbol,market_cap,sector,company_name
13,ASML,5.197802e+11,NaN,ASML Holding NV
18,CCEP,4.254597e+10,NaN,Coca-Cola Europacific Partners plc Ordinary Sh...
70,PDD,1.449885e+11,NaN,PDD Holdings Inc. American Depositary Shares
84,TRI,4.161946e+10,NaN,Thomson Reuters Corporation Common Shares


In [13]:
sector_counts = (universe_df.groupby("sector").size().reset_index(name="n"))
sector_counts

,sector,n
0,AIRCRAFT ENGINES & ENGINE PARTS,1
1,BEVERAGES,2
2,"BIOLOGICAL PRODUCTS, (NO DISGNOSTIC SUBSTANCES)",2
3,BOTTLED & CANNED SOFT DRINKS & CARBONATED WATERS,1
4,CABLE & OTHER PAY TELEVISION SERVICES,2
5,"CANNED, FROZEN & PRESERVD FRUIT, VEG & FOOD SP...",1
6,COMPUTER COMMUNICATIONS EQUIPMENT,1
7,"COMPUTER PERIPHERAL EQUIPMENT, NEC",2
8,COMPUTER STORAGE DEVICES,2
9,CRUDE PETROLEUM & NATURAL GAS,1


In [14]:
sector_counts["possible_pairs"] = (
    sector_counts["n"] *
    (sector_counts["n"] - 1)
) / 2

sector_counts.sort_values(
    "possible_pairs",
    ascending=False
)

,sector,n,possible_pairs
42,SERVICES-PREPACKAGED SOFTWARE,13,78.0
36,SEMICONDUCTORS & RELATED DEVICES,12,66.0
25,PHARMACEUTICAL PREPARATIONS,4,6.0
37,"SERVICES-BUSINESS SERVICES, NEC",4,6.0
38,SERVICES-COMPUTER PROCESSING & DATA PREPARATION,3,3.0
40,"SERVICES-COMPUTER PROGRAMMING, DATA PROCESSING...",3,3.0
2,"BIOLOGICAL PRODUCTS, (NO DISGNOSTIC SUBSTANCES)",2,1.0
7,"COMPUTER PERIPHERAL EQUIPMENT, NEC",2,1.0
39,SERVICES-COMPUTER PROGRAMMING SERVICES,2,1.0
20,MOTOR VEHICLES & PASSENGER CAR BODIES,2,1.0


In [15]:
sector_fixes = {
    "ASML": "SEMICONDUCTORS & RELATED DEVICES",
    "CCEP": "BEVERAGES",
    "PDD": "RETAIL-CATALOG & MAIL-ORDER HOUSES",
    "TRI": "SERVICES-BUSINESS SERVICES, NEC"
}

for symbol, sector in sector_fixes.items():
    universe_df.loc[
        universe_df["symbol"] == symbol,
        "sector"
    ] = sector

In [16]:
universe_df.to_csv("../data/universe_metadata.csv",index=False)

In [17]:
universe_df

,symbol,market_cap,sector,company_name
0,AAPL,4.059188e+12,ELECTRONIC COMPUTERS,Apple Inc.
1,ABNB,7.555724e+10,SERVICES-TO DWELLINGS & OTHER BUILDINGS,"Airbnb, Inc. Class A Common Stock"
2,ADBE,1.148210e+11,SERVICES-PREPACKAGED SOFTWARE,Adobe Inc.
3,ADI,1.565761e+11,SEMICONDUCTORS & RELATED DEVICES,"Analog Devices, Inc."
4,ADP,9.448855e+10,SERVICES-COMPUTER PROCESSING & DATA PREPARATION,Automatic Data Processing
...,...,...,...,...
90,WDAY,4.474945e+10,SERVICES-COMPUTER PROCESSING & DATA PREPARATION,"Workday, Inc. Class A Common Stock"
91,WDC,9.134021e+10,COMPUTER STORAGE DEVICES,Western Digital Corp.
92,WMT,1.020181e+12,RETAIL-VARIETY STORES,Walmart Inc. Common Stock
93,XEL,4.507533e+10,ELECTRIC & OTHER SERVICES COMBINED,"Xcel Energy, Inc."


In [ ]:
universe_df